# v2 Coordination Analysis

This notebook reproduces metric plots from a sample multi-battalion training
run and provides tooling for in-depth analysis of the three coordination
metrics introduced in Epic E2.4:

| Metric | Key | Threshold |
|--------|-----|-----------|
| Flanking ratio | `coordination/flanking_ratio` | > 0.3 |
| Fire concentration | `coordination/fire_concentration` | — |
| Mutual support score | `coordination/mutual_support_score` | — |

**Usage**  
Run all cells top-to-bottom.  The *Synthetic data* section generates a
short simulated episode so the notebook works without a live W&B run.
Replace that section with a real W&B pull when a training run is available.

In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to sys.path so local imports work from the notebooks/ dir
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

## 1 — Metric Definitions

Import the three coordination metric functions from the project's metrics
module and verify they are importable.

In [ ]:
from envs.metrics.coordination import (
    flanking_ratio,
    fire_concentration,
    mutual_support_score,
    compute_all,
)
from envs.sim.battalion import Battalion
from envs.multi_battalion_env import MultiBattalionEnv

print("✅ Imports OK")

## 2 — Synthetic Episode Data

Run a short random-action episode in `MultiBattalionEnv` and collect the
per-step coordination metrics.  Replace this section with a W&B API pull
once a real training run is available.

In [ ]:
SEED = 42
MAX_STEPS = 300

env = MultiBattalionEnv(n_blue=2, n_red=2)
obs, _ = env.reset(seed=SEED)

history = []
for step_idx in range(MAX_STEPS):
    if not env.agents:
        break
    actions = {a: env.action_space(a).sample() for a in env.agents}
    obs, rewards, terminated, truncated, _ = env.step(actions)

    # Collect per-step metrics
    metrics = env.get_coordination_metrics()
    metrics["step"] = step_idx
    metrics["total_reward"] = float(sum(rewards.values()))
    history.append(metrics)

    if all(terminated.get(a, False) or truncated.get(a, False) for a in list(terminated) + list(truncated)):
        break

env.close()
print(f"Episode finished after {len(history)} steps")

## 3 — Metric Time-Series Plots

In [ ]:
steps = np.array([h["step"] for h in history])
fr  = np.array([h["coordination/flanking_ratio"] for h in history])
fc  = np.array([h["coordination/fire_concentration"] for h in history])
mss = np.array([h["coordination/mutual_support_score"] for h in history])

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
fig.suptitle("Coordination Metrics — Episode Time-Series", fontsize=14)

WINDOW = 10  # rolling-mean smoothing window

for ax, values, label, color, threshold in [
    (axes[0], fr,  "flanking_ratio",        "steelblue",  0.3),
    (axes[1], fc,  "fire_concentration",     "darkorange", None),
    (axes[2], mss, "mutual_support_score",   "seagreen",   None),
]:
    ax.plot(steps, values, alpha=0.3, color=color, linewidth=0.8)
    if len(values) >= WINDOW:
        smooth = np.convolve(values, np.ones(WINDOW) / WINDOW, mode="valid")
        ax.plot(steps[WINDOW - 1:], smooth, color=color, linewidth=1.8,
                label=f"{label} (rolling mean)")
    if threshold is not None:
        ax.axhline(threshold, color="red", linestyle="--", linewidth=1,
                   label=f"threshold = {threshold}")
    ax.set_ylabel(label, fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Step")
plt.tight_layout()
plt.savefig("coordination_timeseries.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Saved coordination_timeseries.png")

## 4 — Episode Summary Statistics

In [ ]:
print("Episode coordination summary")
print("=" * 40)
for name, arr in [
    ("flanking_ratio",       fr),
    ("fire_concentration",   fc),
    ("mutual_support_score", mss),
]:
    print(
        f"  {name:<28}  "
        f"mean={arr.mean():.3f}  "
        f"std={arr.std():.3f}  "
        f"min={arr.min():.3f}  "
        f"max={arr.max():.3f}"
    )

meaningful_flanking_pct = (fr > 0.3).mean() * 100
print(f"\nSteps with flanking_ratio > 0.3: {meaningful_flanking_pct:.1f}%")

## 5 — Distribution Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("Coordination Metric Distributions", fontsize=13)

for ax, values, label, color in [
    (axes[0], fr,  "flanking_ratio",        "steelblue"),
    (axes[1], fc,  "fire_concentration",     "darkorange"),
    (axes[2], mss, "mutual_support_score",   "seagreen"),
]:
    ax.hist(values, bins=20, range=(0, 1), color=color, alpha=0.75, edgecolor="white")
    ax.axvline(values.mean(), color="black", linestyle="--", linewidth=1.2,
               label=f"mean = {values.mean():.2f}")
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("coordination_distributions.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Saved coordination_distributions.png")

## 6 — (Optional) Pull from W&B

Replace the synthetic episode above with a real training run by uncommenting
and adapting the cells below.  Requires `wandb` to be installed and
authenticated.

In [ ]:
# import wandb
#
# api = wandb.Api()
# run = api.run("<entity>/<project>/<run_id>")
#
# history_df = run.history(
#     keys=[
#         "coordination/flanking_ratio",
#         "coordination/fire_concentration",
#         "coordination/mutual_support_score",
#         "rollout/win_rate",
#     ]
# )
# history_df.head()